In [1]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Path to your dataset
data_dir = "Blood cell Cancer [ALL]"
img_size = (128, 128)
batch_size = 32

# Data Augmentation and Preprocessing
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True,
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Build CNN Model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(train_data.num_classes, activation='softmax')
])

# Compile the Model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the Model
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

# Evaluate
loss, acc = model.evaluate(val_data)
print(f"Validation Accuracy: {acc*100:.2f}%")

# Classification Report
val_preds = model.predict(val_data)
y_pred = val_preds.argmax(axis=1)
y_true = val_data.classes
class_labels = list(val_data.class_indices.keys())

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))


Found 2595 images belonging to 4 classes.
Found 647 images belonging to 4 classes.


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 169s 2s/step - accuracy: 0.4707 - loss: 1.1660 - val_accuracy: 0.5997 - val_loss: 1.0855
Epoch 2/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 108s 1s/step - accuracy: 0.7628 - loss: 0.6161 - val_accuracy: 0.6043 - val_loss: 1.0910
Epoch 3/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 94s 1s/step - accuracy: 0.7906 - loss: 0.5150 - val_accuracy: 0.6924 - val_loss: 0.8458
Epoch 4/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 90s 1s/step - accuracy: 0.8437 - loss: 0.4138 - val_accuracy: 0.5595 - val_loss: 1.0361
Epoch 5/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 62s 760ms/step - accuracy: 0.8114 - loss: 0.4634 - val_accuracy: 0.6692 - val_loss: 0.8918
Epoch 6/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 76s 927ms/step - accuracy: 0.8459 - loss: 0.3922 - val_accuracy: 0.6692 - val_loss: 0.7638
Epoch 7/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - accuracy: 0.8798 - loss: 0.3290 - val_accuracy: 0.6986 - val_loss: 0.8045
Epoch 8/10
82/82 ━━━━━━━━━━━━━━━━━━━━ 75s 913ms/step - accuracy: 0.8609 - loss: 0.3620 - val_accuracy: 0.7032

In [2]:
model.save('blood_cells_cancer.keras')